# RodPrep — exploration

Notebook d'exploration de l'étape 1 : extraction du récap Excel ROD et construction de la table hôtel.

Objectif : visualiser les entrées, les étapes intermédiaires et remplir `../Output/`.

In [86]:
from pathlib import Path
import sys

import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)

ROOT = Path.cwd().resolve()
while ROOT.name != "RodPrep" and ROOT.parent != ROOT:
    ROOT = ROOT.parent
PREPARE = ROOT.parent
PROJECT = PREPARE.parent

sys.path.insert(0, str(PROJECT))
sys.path.insert(0, str(ROOT / "Src"))

INPUT_DIR = ROOT / "Input"
OUTPUT_DIR = ROOT / "Output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## 1. Entrée — récap Excel et registre identité

In [87]:
from rod_ia.config.settings import get_settings
from rod_ia.domain.repositories.identity_registry import HotelIdentityRegistry
from rod_prep.prep import RodPrep

settings = get_settings(PROJECT)
prep = RodPrep(INPUT_DIR, OUTPUT_DIR, settings.identity_registry_path)

recap_path = prep.seed_input_from_sources()
print("Fichier récap :", recap_path)

registry = HotelIdentityRegistry(settings.identity_registry_path)
registry_df = pd.DataFrame([r.to_dict() for r in registry.all_records()])
print(f"Registre identité : {len(registry_df)} hôtels")
display(registry_df.head(3))
registry_df[["hotel_id", "name_ventes", "brand", "city", "nb_chambres"]].head(10)

Fichier récap : /media/laghmari/ssd-data/dev/hotels/prepare/RodPrep/Input/recapitulatif_rod.xlsx
Registre identité : 8 hôtels


,hotel_id,brand,city,name_display,name_ventes,name_rod,aliases,lat_canonical,lon_canonical,geo_source,lat_rod,lon_rod,lat_nominatim,lon_nominatim,has_sales,has_rod,nb_chambres
0,ibis-budget-nice,IBIS BUDGET,Nice,Ibis budget Nice Californie,Ibis budget Nice,Nice Californie,"[Ibis Budget Nice, IBIS BUDGET Nice]",43.710000,7.260000,nominatim,None,None,43.689258,7.240379,True,True,129.0
1,ibis-budget-strasbourg,IBIS BUDGET,Strasbourg,Ibis budget Strasbourg Centre République,Ibis budget Strasbourg Centre République,Strasbourg République,"[Ibis Budget Strasbourg, Ibis budget Strasbourg]",NaN,NaN,None,None,None,NaN,NaN,True,True,97.0
2,ibis-styles-roissy-cdg,IBIS STYLES,Roissy,Ibis Styles Roissy CDG,None,Roissy CDG,[],49.007078,2.520403,nominatim,None,None,49.007078,2.520403,False,True,309.0


,hotel_id,name_ventes,brand,city,nb_chambres
0,ibis-budget-nice,Ibis budget Nice,IBIS BUDGET,Nice,129.0
1,ibis-budget-strasbourg,Ibis budget Strasbourg Centre République,IBIS BUDGET,Strasbourg,97.0
2,ibis-styles-roissy-cdg,None,IBIS STYLES,Roissy,309.0
3,novotel-megeve,Novotel Megève Mont-Blanc,NOVOTEL,Megève,572.0
4,novotel-paris-tour-eiffel,Novotel Paris Tour Eiffel,NOVOTEL,Paris,764.0
5,mercure-montmartre,Mercure Paris Montmartre Sacré-Cœur,MERCURE,Paris,305.0
6,mercure-boulogne,None,MERCURE,Boulogne-Billancourt,191.0
7,novotel-porte-italie,Novotel Porte d'Italie,NOVOTEL,Paris,NaN


## 2. Extraction longue — une ligne par variable × hôtel

In [88]:
from rod_ia.domain.services.rod_recap_extractor import RodRecapExtractor

extractor = RodRecapExtractor(
    recap_path=recap_path,
    identity_registry=registry,
    output_path=OUTPUT_DIR / "rod_recap",
)

long_df = extractor.extract_long()
print(f"Format long : {long_df.shape[0]} lignes × {long_df.shape[1]} colonnes")
long_df.head(12)

Format long : 938 lignes × 9 colonnes


,hotel_id,recap_column,row,etape,sous_etape,data_label,field_key,field_type_hint,raw_value
0,ibis-budget-nice,NICE,4,0 - PAGE DE CONNEXION,ID,CODE H,0_page_de_connexion_id_code_h,categorical,H2075
1,ibis-budget-strasbourg,STRASBOURG,4,0 - PAGE DE CONNEXION,ID,CODE H,0_page_de_connexion_id_code_h,categorical,HB6A3
2,ibis-styles-roissy-cdg,PARIS CDG,4,0 - PAGE DE CONNEXION,ID,CODE H,0_page_de_connexion_id_code_h,categorical,H0815
3,novotel-megeve,MEGEVE,4,0 - PAGE DE CONNEXION,ID,CODE H,0_page_de_connexion_id_code_h,categorical,HB5I0
4,novotel-paris-tour-eiffel,TOUR EIFFEL,4,0 - PAGE DE CONNEXION,ID,CODE H,0_page_de_connexion_id_code_h,categorical,H3546
5,mercure-montmartre,MONTMARTRE,4,0 - PAGE DE CONNEXION,ID,CODE H,0_page_de_connexion_id_code_h,categorical,H0373
6,mercure-boulogne,BOULOGNE,4,0 - PAGE DE CONNEXION,ID,CODE H,0_page_de_connexion_id_code_h,categorical,H6188
7,ibis-budget-nice,NICE,5,0 - PAGE DE CONNEXION,ID,NOM DE L'HOTEL,0_page_de_connexion_id_nom_de_l_hotel,categorical,IBIS BUDGET NICE CALIFORNIE
8,ibis-budget-strasbourg,STRASBOURG,5,0 - PAGE DE CONNEXION,ID,NOM DE L'HOTEL,0_page_de_connexion_id_nom_de_l_hotel,categorical,IBIS BUDGET STRASBOURG REPUBLIQUE
9,ibis-styles-roissy-cdg,PARIS CDG,5,0 - PAGE DE CONNEXION,ID,NOM DE L'HOTEL,0_page_de_connexion_id_nom_de_l_hotel,categorical,IBIS STYLES ROISSY CDG


## 3. Format wide — features `d_recap_*` par hôtel

In [89]:
wide_df = extractor.extract_wide()
print(f"Format wide : {wide_df.shape[0]} hôtels × {wide_df.shape[1]} colonnes")
wide_df.head()

Format wide : 7 hôtels × 90 colonnes


,hotel_id,d_recap_0_page_de_connexion_localisation_geo_adresse_postale_1,d_recap_0_page_de_connexion_localisation_geo_adresse_postale_2,d_recap_0_page_de_connexion_localisation_geo_code_postal,d_recap_0_page_de_connexion_localisation_geo_latitude,d_recap_0_page_de_connexion_localisation_geo_longitude,d_recap_0_page_de_connexion_localisation_geo_ville,d_recap_2_services_equipements_dispo_dans_le_lobby_assises,d_recap_2_services_equipements_dispo_dans_le_lobby_bouilloire,d_recap_2_services_equipements_dispo_dans_le_lobby_fontaine_a_eau,d_recap_2_services_equipements_dispo_dans_le_lobby_machine_a_cafe,d_recap_2_services_equipements_dispo_dans_le_lobby_micro_ondes,d_recap_2_services_equipements_dispo_dans_le_lobby_vitrine_refrigeree,d_recap_2_services_equipements_f_b_bar,d_recap_2_services_equipements_f_b_horaires_d_ouverture,d_recap_2_services_equipements_f_b_horaires_d_ouverture_r43,d_recap_2_services_equipements_f_b_jours_d_ouverture,d_recap_2_services_equipements_f_b_jours_d_ouverture_r44,d_recap_2_services_equipements_f_b_minibar,d_recap_2_services_equipements_f_b_mois_d_ouverture,d_recap_2_services_equipements_f_b_mois_d_ouverture_r45,d_recap_2_services_equipements_f_b_restaurant,d_recap_2_services_equipements_f_b_room_service,d_recap_2_services_equipements_non_f_b_piscine,d_recap_2_services_equipements_non_f_b_salle_de_sport,d_recap_2_services_equipements_non_f_b_salles_de_reunion,d_recap_2_services_equipements_non_f_b_spa,d_recap_3_profil_de_vos_clients_affaires_affaires_pct,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_f_b_boissons_alcoolisees,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_f_b_boissons_non_alcoolisees,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_f_b_epicerie_fine,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_f_b_produits_sales_frais,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_f_b_produits_sales_secs,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_f_b_produits_sucres_frais,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_f_b_produits_sucres_secs,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_non_f_b_accessoires,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_non_f_b_articles_pour_enfants,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_non_f_b_hygiene,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_non_f_b_pret_a_porter,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_non_f_b_produits_cosmetiques,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_non_f_b_produits_sos,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_non_f_b_souvenirs,d_recap_3_profil_de_vos_clients_loisirs_loisirs_pct,d_recap_3_profil_de_vos_clients_loisirs_top_1_amis,d_recap_3_profil_de_vos_clients_loisirs_top_1_couples,d_recap_3_profil_de_vos_clients_loisirs_top_1_familles,d_recap_3_profil_de_vos_clients_national_vs_inter_international_pct,d_recap_3_profil_de_vos_clients_national_vs_inter_national_pct,d_recap_corner_autre_emplacement_dispo_metres_carres_estimation,d_recap_corner_autre_emplacement_dispo_metres_lineaires_estimation,d_recap_corner_autre_emplacement_dispo_xpct_de_mes_clients_passent_devant,d_recap_corner_corner_de_vente_actuel_metres_carres_estimation,d_recap_corner_corner_de_vente_actuel_metres_lineaires_estimation,d_recap_corner_corner_de_vente_actuel_votre_hotel_dispose_t_il_deja_d_un,d_recap_corner_equipements_disponibles_alimentation_electrique_r124,d_recap_corner_equipements_disponibles_internet_filaire_prise_rj45_r123,d_recap_corner_equipements_disponibles_videosurveillance_r125,d_recap_corner_equipements_disponibles_wifi_r122,d_recap_corner_votre_corner_actuel_offre_f_b_caisse_code_barres,d_recap_corner_votre_corner_actuel_offre_f_b_distributeur_auto,d_recap_corner_votre_corner_actuel_offre_f_b_frigo_connecte,d_recap_corner_votre_corner_actuel_offre_f_b_hotel_staff,d_recap_corner_votre_corner_actuel_offre_f_b_liste_des_produits_f_b,d_recap_corner_votre_corner_actuel_offre_f_b_reception,d_recap_corner_votre_corner_ac

## 4. Table de liaison `hotel_lookup`

In [90]:
hotel_lookup = prep.run()  # persiste aussi rod_features + hotel_lookup
print(f"hotel_lookup : {hotel_lookup.shape}")
hotel_lookup.head()

hotel_lookup : (8, 95)


,hotel_code,hotel_name,nom_hotel,hotel_brand,hotel_city,nb_chambres,d_recap_0_page_de_connexion_localisation_geo_adresse_postale_1,d_recap_0_page_de_connexion_localisation_geo_adresse_postale_2,d_recap_0_page_de_connexion_localisation_geo_code_postal,d_recap_0_page_de_connexion_localisation_geo_ville,d_recap_2_services_equipements_dispo_dans_le_lobby_assises,d_recap_2_services_equipements_dispo_dans_le_lobby_bouilloire,d_recap_2_services_equipements_dispo_dans_le_lobby_fontaine_a_eau,d_recap_2_services_equipements_dispo_dans_le_lobby_machine_a_cafe,d_recap_2_services_equipements_dispo_dans_le_lobby_micro_ondes,d_recap_2_services_equipements_dispo_dans_le_lobby_vitrine_refrigeree,d_recap_2_services_equipements_f_b_bar,d_recap_2_services_equipements_f_b_horaires_d_ouverture,d_recap_2_services_equipements_f_b_horaires_d_ouverture_r43,d_recap_2_services_equipements_f_b_jours_d_ouverture,d_recap_2_services_equipements_f_b_jours_d_ouverture_r44,d_recap_2_services_equipements_f_b_minibar,d_recap_2_services_equipements_f_b_mois_d_ouverture,d_recap_2_services_equipements_f_b_mois_d_ouverture_r45,d_recap_2_services_equipements_f_b_restaurant,d_recap_2_services_equipements_f_b_room_service,d_recap_2_services_equipements_non_f_b_piscine,d_recap_2_services_equipements_non_f_b_salle_de_sport,d_recap_2_services_equipements_non_f_b_salles_de_reunion,d_recap_2_services_equipements_non_f_b_spa,d_recap_3_profil_de_vos_clients_affaires_affaires_pct,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_f_b_boissons_alcoolisees,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_f_b_boissons_non_alcoolisees,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_f_b_epicerie_fine,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_f_b_produits_sales_frais,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_f_b_produits_sales_secs,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_f_b_produits_sucres_frais,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_f_b_produits_sucres_secs,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_non_f_b_accessoires,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_non_f_b_articles_pour_enfants,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_non_f_b_hygiene,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_non_f_b_pret_a_porter,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_non_f_b_produits_cosmetiques,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_non_f_b_produits_sos,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_non_f_b_souvenirs,d_recap_3_profil_de_vos_clients_loisirs_loisirs_pct,d_recap_3_profil_de_vos_clients_loisirs_top_1_amis,d_recap_3_profil_de_vos_clients_loisirs_top_1_couples,d_recap_3_profil_de_vos_clients_loisirs_top_1_familles,d_recap_3_profil_de_vos_clients_national_vs_inter_international_pct,d_recap_3_profil_de_vos_clients_national_vs_inter_national_pct,d_recap_corner_autre_emplacement_dispo_metres_carres_estimation,d_recap_corner_autre_emplacement_dispo_metres_lineaires_estimation,d_recap_corner_autre_emplacement_dispo_xpct_de_mes_clients_passent_devant,d_recap_corner_corner_de_vente_actuel_metres_carres_estimation,d_recap_corner_corner_de_vente_actuel_metres_lineaires_estimation,d_recap_corner_corner_de_vente_actuel_votre_hotel_dispose_t_il_deja_d_un,d_recap_corner_equipements_disponibles_alimentation_electrique_r124,d_recap_corner_equipements_disponibles_internet_filaire_prise_rj45_r123,d_recap_corner_equipements_disponibles_videosurveillance_r125,d_recap_corner_equipements_disponibles_wifi_r122,d_recap_corner_votre_corner_actuel_offre_f_b_caisse_code_barres,d_recap_corner_votre_corner_actuel_offre_f_b_distributeur_auto,d_recap_corner_votre_corner_actuel_offre_f_b_frigo_connecte,d_recap_corner_votre_corner_actuel_offre_f_b_hotel_staff,d_recap_corner_votre_corner_actuel_offre_f_b_liste_des_produits_f_b,d_recap_corner_votre_corner_actuel_offre_f_b_reception,d_recap_corner_votre_corner_actuel_offre_f_b_snacking_comptoir,d_recap_corner_vot

## 5. Aperçu colonnes récap retenues

In [91]:
recap_cols = [c for c in hotel_lookup.columns if str(c).startswith("d_recap_")]
print(f"{len(recap_cols)} colonnes d_recap_")
if recap_cols:
    hotel_lookup[["hotel_code", "nom_hotel"] + recap_cols[:8]].head()

86 colonnes d_recap_


## 6. Entrée MeteoPrep / ProximityPrep

In [92]:
meteo_input = prep.to_meteo_input()
meteo_input

,hotel_code,hotel_name,hotel_brand,hotel_city,hotel_lat,hotel_lon
0,H2075,Ibis budget Nice Californie,IBIS BUDGET,Nice,43.689186,7.240512
1,HB6A3,Ibis budget Strasbourg Centre République,IBIS BUDGET,Strasbourg,48.591522,7.754599
2,H0815,Ibis Styles Roissy CDG,IBIS STYLES,Roissy,49.006733,2.519843
3,H6188,Mercure Paris Boulogne,MERCURE,Boulogne-Billancourt,48.833827,2.256274
4,H0373,Mercure Paris Montmartre Sacré-Cœur,MERCURE,Paris,48.885048,2.329923
5,HB5I0,Novotel Megève Mont-Blanc,NOVOTEL,Megève,45.859165,6.619055
6,H3546,Novotel Paris Centre Tour Eiffel,NOVOTEL,Paris,48.849778,2.282836


## 7. Fichiers produits dans Output/

In [93]:
for path in sorted(OUTPUT_DIR.glob("*")):
    print(path.name)

brand_descriptives.xlsx
hotel_data.xlsx
hotel_lookup.csv
hotel_lookup.parquet
rod_features.csv
rod_features.parquet
rod_recap.long.csv
rod_recap.schema.json
rod_recap.wide.csv


In [94]:
# df = hotel_lookup

In [95]:
unwanted_prefixes= [
    "d_recap_0_page_de_connexion_localisation_geo_",
    "d_recap_2_services_equipements_",
    "d_recap_3_profil_de_vos_clients_affaires_" ,
    "d_recap_3_profil_de_vos_clients_affaires_",
    "d_recap_3_profil_de_vos_clients_",
    "d_recap_corner_",
    "d_recap_de_controle_parametres_",
    "d_recap_generales_donnees_admin_",
    "d_recap_generales_donnees_chiffrees_",
    ""
]


hotel_lookup.columns = hotel_lookup.columns.str.replace(
    "|".join(unwanted_prefixes), 
    "", 
    regex=True
).str.strip().str.replace(r'_+', '_', regex=True).str.strip('_')

In [96]:
hotel_lookup

,hotel_code,hotel_name,nom_hotel,hotel_brand,hotel_city,nb_chambres,adresse_postale_1,adresse_postale_2,code_postal,ville,dispo_dans_le_lobby_assises,dispo_dans_le_lobby_bouilloire,dispo_dans_le_lobby_fontaine_a_eau,dispo_dans_le_lobby_machine_a_cafe,dispo_dans_le_lobby_micro_ondes,dispo_dans_le_lobby_vitrine_refrigeree,f_b_bar,f_b_horaires_d_ouverture,f_b_horaires_d_ouverture_r43,f_b_jours_d_ouverture,f_b_jours_d_ouverture_r44,f_b_minibar,f_b_mois_d_ouverture,f_b_mois_d_ouverture_r45,f_b_restaurant,f_b_room_service,non_f_b_piscine,non_f_b_salle_de_sport,non_f_b_salles_de_reunion,non_f_b_spa,affaires_pct,besoins_de_vos_clients_f_b_boissons_alcoolisees,besoins_de_vos_clients_f_b_boissons_non_alcoolisees,besoins_de_vos_clients_f_b_epicerie_fine,besoins_de_vos_clients_f_b_produits_sales_frais,besoins_de_vos_clients_f_b_produits_sales_secs,besoins_de_vos_clients_f_b_produits_sucres_frais,besoins_de_vos_clients_f_b_produits_sucres_secs,besoins_de_vos_clients_non_f_b_accessoires,besoins_de_vos_clients_non_f_b_articles_pour_enfants,besoins_de_vos_clients_non_f_b_hygiene,besoins_de_vos_clients_non_f_b_pret_a_porter,besoins_de_vos_clients_non_f_b_produits_cosmetiques,besoins_de_vos_clients_non_f_b_produits_sos,besoins_de_vos_clients_non_f_b_souvenirs,loisirs_loisirs_pct,loisirs_top_1_amis,loisirs_top_1_couples,loisirs_top_1_familles,national_vs_inter_international_pct,national_vs_inter_national_pct,autre_emplacement_dispo_metres_carres_estimation,autre_emplacement_dispo_metres_lineaires_estimation,autre_emplacement_dispo_xpct_de_mes_clients_passent_devant,corner_de_vente_actuel_metres_carres_estimation,corner_de_vente_actuel_metres_lineaires_estimation,corner_de_vente_actuel_votre_hotel_dispose_t_il_deja_d_un,equipements_disponibles_alimentation_electrique_r124,equipements_disponibles_internet_filaire_prise_rj45_r123,equipements_disponibles_videosurveillance_r125,equipements_disponibles_wifi_r122,votre_corner_actuel_offre_f_b_caisse_code_barres,votre_corner_actuel_offre_f_b_distributeur_auto,votre_corner_actuel_offre_f_b_frigo_connecte,votre_corner_actuel_offre_f_b_hotel_staff,votre_corner_actuel_offre_f_b_liste_des_produits_f_b,votre_corner_actuel_offre_f_b_reception,votre_corner_actuel_offre_f_b_snacking_comptoir,votre_corner_actuel_offre_non_f_b_armoire_connectee,votre_corner_actuel_offre_non_f_b_caisse_code_barres,votre_corner_actuel_offre_non_f_b_distributeur_auto,votre_corner_actuel_offre_non_f_b_hotel_staff,votre_corner_actuel_offre_non_f_b_liste_des_produits_non_f,votre_corner_actuel_offre_non_f_b_reception,metres_lineaires_dedies_a_v,moyen_de_guests_par_chambre,nb_de_chambres,to_annuel_moyen,contrat_signe_annee,contrat_type,derniere_reno_hotel,derniere_reno_lobby,dom_dof,marque,nb_de_chambres,pms,proprietaire,to_annuel,to_le_plus_bas_mois,to_le_plus_bas_taux,to_le_plus_haut_mois,to_le_plus_haut_taux,hotel_lat,hotel_lon,hotel_geo_source
0,H2075,Ibis budget Nice Californie,Ibis budget Nice,IBIS BUDGET,Nice,129.0,58-60 AVENUE DE LA CALIFORNIE,NaN,06200,NICE,1.0,NaN,1.0,1.0,1.0,1.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,0.0,4.0,0.0,NaN,NaN,1.0,NaN,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,NaN,1.0,1.0,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.0,3.0,1.0,NaN,NaN,NaN,NaN,0.0,1.0,0.0,NaN,1.0,0.0,0.0,0.0,0.0,1.0,NaN,0.0,0.0,6.0,NaN,129.0,NaN,NaN,FRANCHISE,NaN,NaN,Marion BOROT,IBIS BUDGET,129.0,FOLS,FAMILLE FARINES,NaN,NaN,NaN,NaN,NaN,43.689186,7.240512,recap
1,HB6A3,Ibis budget Strasbourg Centre République,Ibis budget Strasbourg Centre République,IBIS BUDGET,Strasbourg,97.0,23A RUE OBERLIN,NaN,67000,STRASBOURG,1.0,0.0,1.0,1.0,0.0,0.0,0.0,16h - 00h,NaN,MARDI AU SAMEDI,NaN,1.0,TOUS,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.4,NaN,1.0,NaN,1.0,1.0,1.0,1.0,NaN,NaN,1.0,NaN,NaN,1.0,NaN,0.6,1.0,NaN,NaN,0.4,0.6,3.0,3.0,100.0,2.0,2.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0,1.0,2.0,1.5,97.0,0.70,2020.0,FRANCHISE,2025.0,2025.0,Christophe RUQUEBOEUCHE,IBIS BUDGET,97.0,OPERA,MILLION - CHOREIN,0.70,JANVIER,0.60,DECEMBRE,0.80,48.591522,7.754599,r

In [97]:
map_cols = {'hotel_code': 'hotel_code',
 'hotel_name': 'hotel_name',
#  'nom_hotel': 'nom_hotel',
 'hotel_brand': 'hotel_brand',
 'hotel_city': 'hotel_city',
 'adresse_postale_1': 'hotel_adresse_postale_1',
 'adresse_postale_2': 'hotel_adresse_postale_2',
 'code_postal': 'hotel_code_postal',
#  'ville': 'ville',

 'hotel_lat': 'hotel_lat',
 'hotel_lon': 'hotel_lon',



#   'marque': 'hotel_marque',
 'contrat_signe_annee': 'hotel_contrat_signe_annee',
 'contrat_type': 'hotel_contrat_type',
 'derniere_reno_hotel': 'hotel_derniere_reno',
 'derniere_reno_lobby': 'hotel_lobby_derniere_reno',
#  'dom_dof': 'dom_dof',
#  'pms': 'hotel_pms',
#  'proprietaire': 'hotel_proprietaire',
 'nb_chambres': 'hotel_nb_chambres',
 'to_annuel': 'hotel_to_annuel',
 # 'nb_de_chambres': 'hotel_nb_de_chambres',
#  'moyen_de_guests_par_chambre': 'hotel_moyen_de_guests_par_chambre',
#  'to_annuel_moyen': 'hotel_to_annuel_moyen',

#  'to_le_plus_bas_mois': 'hotel_to_le_plus_bas_mois',
 'to_le_plus_bas_taux': 'hotel_to_le_plus_bas_taux',
#  'to_le_plus_haut_mois': 'hotel_to_le_plus_haut_mois',
 'to_le_plus_haut_taux': 'hotel_to_le_plus_haut_taux',





 'dispo_dans_le_lobby_assises': 'hotel_dispo_dans_lobby_assises',
 'dispo_dans_le_lobby_bouilloire': 'hotel_dispo_dans_lobby_bouilloire',
 'dispo_dans_le_lobby_fontaine_a_eau': 'hotel_dispo_dans_lobby_fontaine_a_eau',
 'dispo_dans_le_lobby_machine_a_cafe': 'hotel_dispo_dans_lobby_machine_a_cafe',
 'dispo_dans_le_lobby_micro_ondes': 'hotel_dispo_dans_lobby_micro_ondes',
 'dispo_dans_le_lobby_vitrine_refrigeree': 'hotel_dispo_dans_lobby_vitrine_refrigeree',
 
 'f_b_bar': 'hotel_f_b_bar',
#  'f_b_horaires_d_ouverture': 'f_b_horaires_d_ouverture',
#  'f_b_horaires_d_ouverture_r43': 'f_b_horaires_d_ouverture_r43',
#  'f_b_jours_d_ouverture': 'f_b_jours_d_ouverture',
#  'f_b_jours_d_ouverture_r44': 'f_b_jours_d_ouverture_r44',
 'f_b_minibar': 'hotel_f_b_minibar',
#  'f_b_mois_d_ouverture': 'f_b_mois_d_ouverture',
#  'f_b_mois_d_ouverture_r45': 'f_b_mois_d_ouverture_r45',
 'f_b_restaurant': 'hotel_f_b_restaurant',
 'f_b_room_service': 'hotel_f_b_room_service',

 'non_f_b_piscine': 'hotel_non_f_b_piscine',
 'non_f_b_salle_de_sport': 'hotel_non_f_b_salle_de_sport',
 'non_f_b_salles_de_reunion': 'hotel_non_f_b_salles_de_reunion',
 'non_f_b_spa': 'hotel_non_f_b_spa',



#  'besoins_de_vos_clients_f_b_boissons_alcoolisees': 'besoins_f_b_boissons_alcoolisees',
#  'besoins_de_vos_clients_f_b_boissons_non_alcoolisees': 'besoins_f_b_boissons_non_alcoolisees',
#  'besoins_de_vos_clients_f_b_epicerie_fine': 'besoins_f_b_epicerie_fine',
#  'besoins_de_vos_clients_f_b_produits_sales_frais': 'besoins_f_b_produits_sales_frais',
#  'besoins_de_vos_clients_f_b_produits_sales_secs': 'besoins_f_b_produits_sales_secs',
#  'besoins_de_vos_clients_f_b_produits_sucres_frais': 'besoins_f_b_produits_sucres_frais',
#  'besoins_de_vos_clients_f_b_produits_sucres_secs': 'besoins_f_b_produits_sucres_secs',

#  'besoins_de_vos_clients_non_f_b_accessoires': 'besoins_non_f_b_accessoires',
#  'besoins_de_vos_clients_non_f_b_articles_pour_enfants': 'besoins_non_f_b_articles_pour_enfants',
#  'besoins_de_vos_clients_non_f_b_hygiene': 'besoins_non_f_b_hygiene',
#  'besoins_de_vos_clients_non_f_b_pret_a_porter': 'besoins_non_f_b_pret_a_porter',
#  'besoins_de_vos_clients_non_f_b_produits_cosmetiques': 'besoins_non_f_b_produits_cosmetiques',
#  'besoins_de_vos_clients_non_f_b_produits_sos': 'besoins_non_f_b_produits_sos',
#  'besoins_de_vos_clients_non_f_b_souvenirs': 'besoins_non_f_b_souvenirs',

 'affaires_pct': 'hotel_affaires_pct',
 'loisirs_loisirs_pct': 'hotel_loisirs_pct',
 'loisirs_top_1_amis': 'hotel_loisirs_top_1_amis',
 'loisirs_top_1_couples': 'hotel_loisirs_top_1_couples',
 'loisirs_top_1_familles': 'hotel_loisirs_top_1_familles',

 'national_vs_inter_international_pct': 'hotel_international_pct',
 'national_vs_inter_national_pct': 'hotel_national_pct',
 
#  'autre_emplacement_dispo_metres_carres_estimation': 'autre_emplacement_dispo_metres_carres_estimation',
#  'autre_emplacement_dispo_metres_lineaires_estimation': 'autre_emplacement_dispo_metres_lineaires_estimation',
#  'autre_emplacement_dispo_xpct_de_mes_clients_passent_devant': 'autre_emplacement_dispo_xpct_de_mes_clients_passent_devant',
  
 'corner_de_vente_actuel_votre_hotel_dispose_t_il_deja_d_un': 'hotel_corner_actuel_existe_deja',
 'corner_de_vente_actuel_metres_lineaires_estimation': 'hotel_corner_de_vente_actuel_metres_lineaires',

 'votre_corner_actuel_offre_f_b_caisse_code_barres': 'hotel_corner_actuel_offre_f_b_caisse_code_barres',
 'votre_corner_actuel_offre_f_b_distributeur_auto': 'hotel_corner_actuel_offre_f_b_distributeur_auto',
 'votre_corner_actuel_offre_f_b_frigo_connecte': 'hotel_corner_actuel_offre_f_b_frigo_connecte',
 'votre_corner_actuel_offre_f_b_reception': 'hotel_corner_actuel_offre_f_b_reception',
 'votre_corner_actuel_offre_f_b_snacking_comptoir': 'hotel_corner_actuel_offre_f_b_snacking_comptoir',
#  'votre_corner_actuel_offre_f_b_hotel_staff': 'hohtel_corner_actuel_offre_f_b_reappro_hotel_staff',

#  'corner_de_vente_actuel_metres_carres_estimation': 'corner_de_vente_actuel_metres_carres_estimation',
 
 

#  'equipements_disponibles_alimentation_electrique_r124': 'equipements_disponibles_alimentation_electrique_r124',
#  'equipements_disponibles_internet_filaire_prise_rj45_r123': 'equipements_disponibles_internet_filaire_prise_rj45_r123',
#  'equipements_disponibles_videosurveillance_r125': 'equipements_disponibles_videosurveillance_r125',
#  'equipements_disponibles_wifi_r122': 'equipements_disponibles_wifi_r122',



#  'votre_corner_actuel_offre_f_b_liste_des_produits_f_b': 'votre_corner_actuel_offre_f_b_liste_des_produits_f_b',
 

 'votre_corner_actuel_offre_non_f_b_armoire_connectee': 'hotel_corner_actuel_offre_non_f_b_armoire_connectee',
 'votre_corner_actuel_offre_non_f_b_caisse_code_barres': 'hotel_corner_actuel_offre_non_f_b_caisse_code_barres',
 'votre_corner_actuel_offre_non_f_b_distributeur_auto': 'hotel_corner_actuel_offre_non_f_b_distributeur_auto',
#  'votre_corner_actuel_offre_non_f_b_hotel_staff': 'hotel_corner_actuel_offre_non_f_b_hotel_staff',
#  'votre_corner_actuel_offre_non_f_b_liste_des_produits_non_f': 'hotel_corner_actuel_offre_non_f_b_liste_des_produits_non_f',
 'votre_corner_actuel_offre_non_f_b_reception': 'hotel_corner_actuel_offre_non_f_b_reception',


 

 'metres_lineaires_dedies_a_v': 'hotel_metres_lineaires_dedies_corner',


#  'hotel_geo_source': 'hotel_geo_source'
 }

In [98]:
hotel_features = (
    hotel_lookup.iloc[:-1][list(map_cols.keys())]
    .rename(columns=map_cols)
    .copy()
)
hotel_features

,hotel_code,hotel_name,hotel_brand,hotel_city,hotel_adresse_postale_1,hotel_adresse_postale_2,hotel_code_postal,hotel_lat,hotel_lon,hotel_contrat_signe_annee,hotel_contrat_type,hotel_derniere_reno,hotel_lobby_derniere_reno,hotel_nb_chambres,hotel_to_annuel,hotel_to_le_plus_bas_taux,hotel_to_le_plus_haut_taux,hotel_dispo_dans_lobby_assises,hotel_dispo_dans_lobby_bouilloire,hotel_dispo_dans_lobby_fontaine_a_eau,hotel_dispo_dans_lobby_machine_a_cafe,hotel_dispo_dans_lobby_micro_ondes,hotel_dispo_dans_lobby_vitrine_refrigeree,hotel_f_b_bar,hotel_f_b_minibar,hotel_f_b_restaurant,hotel_f_b_room_service,hotel_non_f_b_piscine,hotel_non_f_b_salle_de_sport,hotel_non_f_b_salles_de_reunion,hotel_non_f_b_spa,hotel_affaires_pct,hotel_loisirs_pct,hotel_loisirs_top_1_amis,hotel_loisirs_top_1_couples,hotel_loisirs_top_1_familles,hotel_international_pct,hotel_national_pct,hotel_corner_actuel_existe_deja,hotel_corner_de_vente_actuel_metres_lineaires,hotel_corner_actuel_offre_f_b_caisse_code_barres,hotel_corner_actuel_offre_f_b_distributeur_auto,hotel_corner_actuel_offre_f_b_frigo_connecte,hotel_corner_actuel_offre_f_b_reception,hotel_corner_actuel_offre_f_b_snacking_comptoir,hotel_corner_actuel_offre_non_f_b_armoire_connectee,hotel_corner_actuel_offre_non_f_b_caisse_code_barres,hotel_corner_actuel_offre_non_f_b_distributeur_auto,hotel_corner_actuel_offre_non_f_b_reception,hotel_metres_lineaires_dedies_corner
0,H2075,Ibis budget Nice Californie,IBIS BUDGET,Nice,58-60 AVENUE DE LA CALIFORNIE,NaN,06200,43.689186,7.240512,NaN,FRANCHISE,NaN,NaN,129.0,NaN,NaN,NaN,1.0,NaN,1.0,1.0,1.0,1.0,0.0,NaN,0.0,NaN,0.0,0.0,4.0,0.0,NaN,NaN,1.0,NaN,NaN,NaN,NaN,1.0,3.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,6.0
1,HB6A3,Ibis budget Strasbourg Centre République,IBIS BUDGET,Strasbourg,23A RUE OBERLIN,NaN,67000,48.591522,7.754599,2020.0,FRANCHISE,2025.0,2025.0,97.0,0.70,0.60,0.80,1.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.4,0.6,1.0,NaN,NaN,0.4,0.6,1.0,2.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,2.0
2,H0815,Ibis Styles Roissy CDG,IBIS STYLES,Roissy,2 AVENUE HEINZ GLOOR,NaN,95700,49.006733,2.519843,2015.0,FRANCHISE,2015.0,2015.0,309.0,0.95,0.91,0.98,1.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,20.0,80.0,NaN,NaN,1.0,30.0,70.0,1.0,NaN,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,5.0
3,H6188,Mercure Paris Boulogne,MERCURE,Boulogne-Billancourt,37 PLACE RENÉ CLAIR,NaN,92100,48.833827,2.256274,NaN,FRANCHISE,NaN,NaN,191.0,NaN,NaN,NaN,1.0,0.0,1.0,0.0,0.0,1.0,1.0,NaN,2.0,NaN,1.0,1.0,13.0,0.0,NaN,NaN,NaN,1.0,NaN,NaN,NaN,1.0,NaN,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,6.0
4,H0373,Mercure Paris Montmartre Sacré-Cœur,MERCURE,Paris,3 RUE CAULAINCOURT,NaN,75018,48.885048,2.329923,NaN,MANAGÉ,NaN,NaN,305.0,NaN,NaN,NaN,1.0,0.0,0.0,0.0,0.0,1.0,0.0,NaN,1.0,NaN,0.0,1.0,1.0,0.0,NaN,NaN,NaN,1.0,NaN,NaN,NaN,1.0,NaN,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,6.0
5,HB5I0,Novotel Megève Mont-Blanc,NOVOTEL,Megève,1306 ROUTE NATIONALE,LE DOMAINE DE MEZTIVA,74120,45.859165,6.619055,NaN,FRANCHISE,NaN,NaN,572.0,NaN,NaN,NaN,1.0,0.0,0.0,0.0,0.0,0.0,1.0,NaN,1.0,NaN,1.0,2.0,3.0,0.0,NaN,NaN,NaN,NaN,1.0,NaN,NaN,0.0,NaN,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,8.0
6,H3546,Novotel Paris Centre Tour Eiffel,NOVOTEL,Paris,61 QUAI DE GRENELLE,NaN,75015,48.849778,2.282836,1976.0,MANAGÉ,NaN,NaN,764.0,NaN,NaN,NaN,1.0,0.0,1.0,1.0,0.0,0.0,1.0,NaN,3.0,NaN,1.0,1.0,36.0,0.0,NaN,NaN,NaN,NaN,1.0,NaN,NaN,1.0,NaN,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,7.0


In [99]:
imputation_with_average_value = [
"hotel_corner_de_vente_actuel_metres_lineaires",
"hotel_international_pct",
"hotel_national_pct",
"hotel_affaires_pct",
"hotel_to_le_plus_bas_taux",
"hotel_to_le_plus_haut_taux",
"hotel_to_annuel",
"hotel_derniere_reno",
"hotel_lobby_derniere_reno",
"hotel_contrat_signe_annee",
]

In [100]:
import pandas as pd

# Adresses postales : on ne touche pas aux nulls
ADDRESS_COLS = {"hotel_adresse_postale_1", "hotel_adresse_postale_2"}

# Colonnes texte / identifiants : pas d'imputation
INFO_COLS = {
    "hotel_code",
    "hotel_name",
    "hotel_brand",
    "hotel_city",
    "hotel_code_postal",
    "hotel_contrat_type",
}

# Moyenne pour les indicateurs continus identifiés
for col in imputation_with_average_value:
    if col in hotel_features.columns:
        hotel_features[col] = hotel_features[col].fillna(hotel_features[col].mean())

# Tout le reste (numérique) → 0
zero_cols = [
    c
    for c in hotel_features.columns
    if c not in ADDRESS_COLS
    and c not in INFO_COLS
    and c not in imputation_with_average_value
    and pd.api.types.is_numeric_dtype(hotel_features[c])
]
hotel_features[zero_cols] = hotel_features[zero_cols].fillna(0)

remaining_nulls = hotel_features.isna().sum()
remaining_nulls = remaining_nulls[remaining_nulls > 0].sort_values(ascending=False)
print(f"Nulls restants : {int(hotel_features.isna().sum().sum())}")
if not remaining_nulls.empty:
    print(remaining_nulls)
hotel_features

Nulls restants : 6
hotel_adresse_postale_2    6
dtype: int64


,hotel_code,hotel_name,hotel_brand,hotel_city,hotel_adresse_postale_1,hotel_adresse_postale_2,hotel_code_postal,hotel_lat,hotel_lon,hotel_contrat_signe_annee,hotel_contrat_type,hotel_derniere_reno,hotel_lobby_derniere_reno,hotel_nb_chambres,hotel_to_annuel,hotel_to_le_plus_bas_taux,hotel_to_le_plus_haut_taux,hotel_dispo_dans_lobby_assises,hotel_dispo_dans_lobby_bouilloire,hotel_dispo_dans_lobby_fontaine_a_eau,hotel_dispo_dans_lobby_machine_a_cafe,hotel_dispo_dans_lobby_micro_ondes,hotel_dispo_dans_lobby_vitrine_refrigeree,hotel_f_b_bar,hotel_f_b_minibar,hotel_f_b_restaurant,hotel_f_b_room_service,hotel_non_f_b_piscine,hotel_non_f_b_salle_de_sport,hotel_non_f_b_salles_de_reunion,hotel_non_f_b_spa,hotel_affaires_pct,hotel_loisirs_pct,hotel_loisirs_top_1_amis,hotel_loisirs_top_1_couples,hotel_loisirs_top_1_familles,hotel_international_pct,hotel_national_pct,hotel_corner_actuel_existe_deja,hotel_corner_de_vente_actuel_metres_lineaires,hotel_corner_actuel_offre_f_b_caisse_code_barres,hotel_corner_actuel_offre_f_b_distributeur_auto,hotel_corner_actuel_offre_f_b_frigo_connecte,hotel_corner_actuel_offre_f_b_reception,hotel_corner_actuel_offre_f_b_snacking_comptoir,hotel_corner_actuel_offre_non_f_b_armoire_connectee,hotel_corner_actuel_offre_non_f_b_caisse_code_barres,hotel_corner_actuel_offre_non_f_b_distributeur_auto,hotel_corner_actuel_offre_non_f_b_reception,hotel_metres_lineaires_dedies_corner
0,H2075,Ibis budget Nice Californie,IBIS BUDGET,Nice,58-60 AVENUE DE LA CALIFORNIE,NaN,06200,43.689186,7.240512,2003.666667,FRANCHISE,2020.0,2020.0,129.0,0.825,0.755,0.89,1.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,0.0,10.2,0.0,1.0,0.0,0.0,15.2,35.3,1.0,3.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,6.0
1,HB6A3,Ibis budget Strasbourg Centre République,IBIS BUDGET,Strasbourg,23A RUE OBERLIN,NaN,67000,48.591522,7.754599,2020.000000,FRANCHISE,2025.0,2025.0,97.0,0.700,0.600,0.80,1.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.4,0.6,1.0,0.0,0.0,0.4,0.6,1.0,2.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,2.0
2,H0815,Ibis Styles Roissy CDG,IBIS STYLES,Roissy,2 AVENUE HEINZ GLOOR,NaN,95700,49.006733,2.519843,2015.000000,FRANCHISE,2015.0,2015.0,309.0,0.950,0.910,0.98,1.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,20.0,80.0,0.0,0.0,1.0,30.0,70.0,1.0,2.5,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,5.0
3,H6188,Mercure Paris Boulogne,MERCURE,Boulogne-Billancourt,37 PLACE RENÉ CLAIR,NaN,92100,48.833827,2.256274,2003.666667,FRANCHISE,2020.0,2020.0,191.0,0.825,0.755,0.89,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,2.0,0.0,1.0,1.0,13.0,0.0,10.2,0.0,0.0,1.0,0.0,15.2,35.3,1.0,2.5,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,6.0
4,H0373,Mercure Paris Montmartre Sacré-Cœur,MERCURE,Paris,3 RUE CAULAINCOURT,NaN,75018,48.885048,2.329923,2003.666667,MANAGÉ,2020.0,2020.0,305.0,0.825,0.755,0.89,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,10.2,0.0,0.0,1.0,0.0,15.2,35.3,1.0,2.5,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,6.0
5,HB5I0,Novotel Megève Mont-Blanc,NOVOTEL,Megève,1306 ROUTE NATIONALE,LE DOMAINE DE MEZTIVA,74120,45.859165,6.619055,2003.666667,FRANCHISE,2020.0,2020.0,572.0,0.825,0.755,0.89,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,2.0,3.0,0.0,10.2,0.0,0.0,0.0,1.0,15.2,35.3,0.0,2.5,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,8.0
6,H3546,Novotel Paris Centre Tour Eiffel,NOVOTEL,Paris,61 QUAI DE GRENELLE,NaN,75015,48.849778,2.282836,1976.000000,MANAGÉ,2020.0,2020.0,764.0,0.825,0.755,0.89,1.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,3.0,0.0,1.0,1.0,36.0,0.0,10.2,0.0,0.0,0.0,1.0,15.2,35.3,1.0,2.5,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,7.0


In [101]:
# Contrat : dummy FRANCHISE / MANAGE (ou MANAGÉ)
contrat = hotel_features["hotel_contrat_type"].astype(str).str.upper().str.strip()
hotel_features["hotel_contrat_type_franchise"] = (contrat == "FRANCHISE").astype(float)
hotel_features["hotel_contrat_type_manage"] = contrat.str.startswith("MANAG").astype(float)
hotel_features = hotel_features.drop(columns=["hotel_contrat_type"])

# Marque : dummy puis suppression de hotel_brand
brand = hotel_features["hotel_brand"].astype(str).str.upper().str.strip()
hotel_features["hotel_brand_ibis_budget"] = (brand == "IBIS BUDGET").astype(float)
hotel_features["hotel_brand_ibis_styles"] = (brand == "IBIS STYLES").astype(float)
hotel_features["hotel_brand_mercure"] = (brand == "MERCURE").astype(float)
hotel_features["hotel_brand_novotel"] = (brand == "NOVOTEL").astype(float)
#hotel_features = hotel_features.drop(columns=["hotel_brand"])

# Ordre des colonnes : identifiants / adresse, puis numériques
ID_COLS = [
    "hotel_code",
    "hotel_name",
    "hotel_brand",
    "hotel_adresse_postale_1",
    "hotel_adresse_postale_2",
    "hotel_code_postal",
    "hotel_city",
]
numeric_cols = [
    c
    for c in hotel_features.columns
    if c not in ID_COLS and pd.api.types.is_numeric_dtype(hotel_features[c])
]
hotel_features = hotel_features[ID_COLS + numeric_cols]
hotel_features

,hotel_code,hotel_name,hotel_brand,hotel_adresse_postale_1,hotel_adresse_postale_2,hotel_code_postal,hotel_city,hotel_lat,hotel_lon,hotel_contrat_signe_annee,hotel_derniere_reno,hotel_lobby_derniere_reno,hotel_nb_chambres,hotel_to_annuel,hotel_to_le_plus_bas_taux,hotel_to_le_plus_haut_taux,hotel_dispo_dans_lobby_assises,hotel_dispo_dans_lobby_bouilloire,hotel_dispo_dans_lobby_fontaine_a_eau,hotel_dispo_dans_lobby_machine_a_cafe,hotel_dispo_dans_lobby_micro_ondes,hotel_dispo_dans_lobby_vitrine_refrigeree,hotel_f_b_bar,hotel_f_b_minibar,hotel_f_b_restaurant,hotel_f_b_room_service,hotel_non_f_b_piscine,hotel_non_f_b_salle_de_sport,hotel_non_f_b_salles_de_reunion,hotel_non_f_b_spa,hotel_affaires_pct,hotel_loisirs_pct,hotel_loisirs_top_1_amis,hotel_loisirs_top_1_couples,hotel_loisirs_top_1_familles,hotel_international_pct,hotel_national_pct,hotel_corner_actuel_existe_deja,hotel_corner_de_vente_actuel_metres_lineaires,hotel_corner_actuel_offre_f_b_caisse_code_barres,hotel_corner_actuel_offre_f_b_distributeur_auto,hotel_corner_actuel_offre_f_b_frigo_connecte,hotel_corner_actuel_offre_f_b_reception,hotel_corner_actuel_offre_f_b_snacking_comptoir,hotel_corner_actuel_offre_non_f_b_armoire_connectee,hotel_corner_actuel_offre_non_f_b_caisse_code_barres,hotel_corner_actuel_offre_non_f_b_distributeur_auto,hotel_corner_actuel_offre_non_f_b_reception,hotel_metres_lineaires_dedies_corner,hotel_contrat_type_franchise,hotel_contrat_type_manage,hotel_brand_ibis_budget,hotel_brand_ibis_styles,hotel_brand_mercure,hotel_brand_novotel
0,H2075,Ibis budget Nice Californie,IBIS BUDGET,58-60 AVENUE DE LA CALIFORNIE,NaN,06200,Nice,43.689186,7.240512,2003.666667,2020.0,2020.0,129.0,0.825,0.755,0.89,1.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,0.0,10.2,0.0,1.0,0.0,0.0,15.2,35.3,1.0,3.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,6.0,1.0,0.0,1.0,0.0,0.0,0.0
1,HB6A3,Ibis budget Strasbourg Centre République,IBIS BUDGET,23A RUE OBERLIN,NaN,67000,Strasbourg,48.591522,7.754599,2020.000000,2025.0,2025.0,97.0,0.700,0.600,0.80,1.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.4,0.6,1.0,0.0,0.0,0.4,0.6,1.0,2.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,2.0,1.0,0.0,1.0,0.0,0.0,0.0
2,H0815,Ibis Styles Roissy CDG,IBIS STYLES,2 AVENUE HEINZ GLOOR,NaN,95700,Roissy,49.006733,2.519843,2015.000000,2015.0,2015.0,309.0,0.950,0.910,0.98,1.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,20.0,80.0,0.0,0.0,1.0,30.0,70.0,1.0,2.5,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,5.0,1.0,0.0,0.0,1.0,0.0,0.0
3,H6188,Mercure Paris Boulogne,MERCURE,37 PLACE RENÉ CLAIR,NaN,92100,Boulogne-Billancourt,48.833827,2.256274,2003.666667,2020.0,2020.0,191.0,0.825,0.755,0.89,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,2.0,0.0,1.0,1.0,13.0,0.0,10.2,0.0,0.0,1.0,0.0,15.2,35.3,1.0,2.5,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,6.0,1.0,0.0,0.0,0.0,1.0,0.0
4,H0373,Mercure Paris Montmartre Sacré-Cœur,MERCURE,3 RUE CAULAINCOURT,NaN,75018,Paris,48.885048,2.329923,2003.666667,2020.0,2020.0,305.0,0.825,0.755,0.89,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,10.2,0.0,0.0,1.0,0.0,15.2,35.3,1.0,2.5,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,6.0,0.0,1.0,0.0,0.0,1.0,0.0
5,HB5I0,Novotel Megève Mont-Blanc,NOVOTEL,1306 ROUTE NATIONALE,LE DOMAINE DE MEZTIVA,74120,Megève,45.859165,6.619055,2003.666667,2020.0,2020.0,572.0,0.825,0.755,0.89,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,2.0,3.0,0.0,10.2,0.0,0.0,0.0,1.0,15.2,35.3,0.0,2.5,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,8.0,1.0,0.0,0.0,0.0,0.0,1.0
6,H3546,Novotel Paris Centre Tour Eiffel,NOVOTEL,61 QUAI DE GRENELLE,NaN,75015,Paris,48.849778,2.282836,1976.000000,2020.0,2020.0,764.0,0.825,0.755,0.89,1.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,3.0,0.0,1.0,1.0,36.0,0.0,10.2,0.0,0.0,0.0,1.0,15.2,35.3,1.0,2.5,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,7.0,0.0,1.0,0.0,0.0,0.0,1.0


In [102]:
hotel_features

,hotel_code,hotel_name,hotel_brand,hotel_adresse_postale_1,hotel_adresse_postale_2,hotel_code_postal,hotel_city,hotel_lat,hotel_lon,hotel_contrat_signe_annee,hotel_derniere_reno,hotel_lobby_derniere_reno,hotel_nb_chambres,hotel_to_annuel,hotel_to_le_plus_bas_taux,hotel_to_le_plus_haut_taux,hotel_dispo_dans_lobby_assises,hotel_dispo_dans_lobby_bouilloire,hotel_dispo_dans_lobby_fontaine_a_eau,hotel_dispo_dans_lobby_machine_a_cafe,hotel_dispo_dans_lobby_micro_ondes,hotel_dispo_dans_lobby_vitrine_refrigeree,hotel_f_b_bar,hotel_f_b_minibar,hotel_f_b_restaurant,hotel_f_b_room_service,hotel_non_f_b_piscine,hotel_non_f_b_salle_de_sport,hotel_non_f_b_salles_de_reunion,hotel_non_f_b_spa,hotel_affaires_pct,hotel_loisirs_pct,hotel_loisirs_top_1_amis,hotel_loisirs_top_1_couples,hotel_loisirs_top_1_familles,hotel_international_pct,hotel_national_pct,hotel_corner_actuel_existe_deja,hotel_corner_de_vente_actuel_metres_lineaires,hotel_corner_actuel_offre_f_b_caisse_code_barres,hotel_corner_actuel_offre_f_b_distributeur_auto,hotel_corner_actuel_offre_f_b_frigo_connecte,hotel_corner_actuel_offre_f_b_reception,hotel_corner_actuel_offre_f_b_snacking_comptoir,hotel_corner_actuel_offre_non_f_b_armoire_connectee,hotel_corner_actuel_offre_non_f_b_caisse_code_barres,hotel_corner_actuel_offre_non_f_b_distributeur_auto,hotel_corner_actuel_offre_non_f_b_reception,hotel_metres_lineaires_dedies_corner,hotel_contrat_type_franchise,hotel_contrat_type_manage,hotel_brand_ibis_budget,hotel_brand_ibis_styles,hotel_brand_mercure,hotel_brand_novotel
0,H2075,Ibis budget Nice Californie,IBIS BUDGET,58-60 AVENUE DE LA CALIFORNIE,NaN,06200,Nice,43.689186,7.240512,2003.666667,2020.0,2020.0,129.0,0.825,0.755,0.89,1.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,0.0,10.2,0.0,1.0,0.0,0.0,15.2,35.3,1.0,3.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,6.0,1.0,0.0,1.0,0.0,0.0,0.0
1,HB6A3,Ibis budget Strasbourg Centre République,IBIS BUDGET,23A RUE OBERLIN,NaN,67000,Strasbourg,48.591522,7.754599,2020.000000,2025.0,2025.0,97.0,0.700,0.600,0.80,1.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.4,0.6,1.0,0.0,0.0,0.4,0.6,1.0,2.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,2.0,1.0,0.0,1.0,0.0,0.0,0.0
2,H0815,Ibis Styles Roissy CDG,IBIS STYLES,2 AVENUE HEINZ GLOOR,NaN,95700,Roissy,49.006733,2.519843,2015.000000,2015.0,2015.0,309.0,0.950,0.910,0.98,1.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,20.0,80.0,0.0,0.0,1.0,30.0,70.0,1.0,2.5,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,5.0,1.0,0.0,0.0,1.0,0.0,0.0
3,H6188,Mercure Paris Boulogne,MERCURE,37 PLACE RENÉ CLAIR,NaN,92100,Boulogne-Billancourt,48.833827,2.256274,2003.666667,2020.0,2020.0,191.0,0.825,0.755,0.89,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,2.0,0.0,1.0,1.0,13.0,0.0,10.2,0.0,0.0,1.0,0.0,15.2,35.3,1.0,2.5,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,6.0,1.0,0.0,0.0,0.0,1.0,0.0
4,H0373,Mercure Paris Montmartre Sacré-Cœur,MERCURE,3 RUE CAULAINCOURT,NaN,75018,Paris,48.885048,2.329923,2003.666667,2020.0,2020.0,305.0,0.825,0.755,0.89,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,10.2,0.0,0.0,1.0,0.0,15.2,35.3,1.0,2.5,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,6.0,0.0,1.0,0.0,0.0,1.0,0.0
5,HB5I0,Novotel Megève Mont-Blanc,NOVOTEL,1306 ROUTE NATIONALE,LE DOMAINE DE MEZTIVA,74120,Megève,45.859165,6.619055,2003.666667,2020.0,2020.0,572.0,0.825,0.755,0.89,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,2.0,3.0,0.0,10.2,0.0,0.0,0.0,1.0,15.2,35.3,0.0,2.5,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,8.0,1.0,0.0,0.0,0.0,0.0,1.0
6,H3546,Novotel Paris Centre Tour Eiffel,NOVOTEL,61 QUAI DE GRENELLE,NaN,75015,Paris,48.849778,2.282836,1976.000000,2020.0,2020.0,764.0,0.825,0.755,0.89,1.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,3.0,0.0,1.0,1.0,36.0,0.0,10.2,0.0,0.0,0.0,1.0,15.2,35.3,1.0,2.5,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,7.0,0.0,1.0,0.0,0.0,0.0,1.0


In [103]:
hotel_features.to_excel("../Output/hotel_data.xlsx", index = False)

In [104]:
# hotel brand data

In [105]:
hotel_brand_feautres = pd.read_excel("../Input/hotel_brand_data.xlsx")

In [109]:
hotel_brand_feautres.rename({col : col.lower() for col in hotel_brand_feautres.columns}, axis=1)


,marque,nb_hotels,pct_parc,nb_ch_0_49,pct_brand_ch_0_49,pct_parc_ch_0_49,nb_ch_50_99,pct_brand_ch_50_99,pct_parc_ch_50_99,nb_ch_100_149,pct_brand_ch_100_149,pct_parc_ch_100_149,nb_ch_150_199,pct_brand_ch_150_199,pct_parc_ch_150_199,nb_ch_200_249,pct_brand_ch_200_249,pct_parc_ch_200_249,nb_ch_250_299,pct_brand_ch_250_299,pct_parc_ch_250_299,nb_ch_300_plus,pct_brand_ch_300_plus,pct_parc_ch_300_plus,nb_resto_0,pct_brand_resto_0,nb_resto_1,pct_brand_resto_1,nb_resto_2,pct_brand_resto_2,nb_resto_3,pct_brand_resto_3,nb_resto_total,nb_bar_0,pct_brand_bar_0,nb_bar_1,pct_brand_bar_1,nb_bar_2,pct_brand_bar_2,nb_bar_3,pct_brand_bar_3,nb_bar_total
0,IBIS BUDGET,342,0.254655,45,0.131579,0.033508,252,0.736842,0.187640,31,0.090643,0.023083,7,0.020468,0.005211,3,0.008772,0.002232,3,0.008772,0.002232,1,0.002924,0.000744,314,0.918129,28,0.081871,0,0.000000,0,0.000000,342,319,0.932749,23,0.067251,0,0.000000,0,0.000000,342
1,IBIS STYLES,267,0.198809,60,0.224719,0.044677,174,0.651685,0.129561,26,0.097378,0.019360,3,0.011236,0.002232,1,0.003745,0.000744,0,0.000000,0.000000,3,0.011236,0.002232,181,0.677903,81,0.303371,4,0.014981,1,0.003745,267,48,0.179775,218,0.816479,1,0.003745,0,0.000000,267
2,IBIS,362,0.269546,45,0.124309,0.033508,247,0.682320,0.183917,45,0.124309,0.033508,11,0.030387,0.008191,3,0.008287,0.002232,4,0.011050,0.002978,7,0.019337,0.005211,181,0.500000,176,0.486188,4,0.011050,1,0.002762,362,15,0.041436,341,0.941989,6,0.016575,0,0.000000,362
3,MERCURE,255,0.189873,28,0.109804,0.020850,155,0.607843,0.115413,45,0.176471,0.033508,15,0.058824,0.011167,5,0.019608,0.003722,0,0.000000,0.000000,7,0.027451,0.005211,106,0.415686,140,0.549020,8,0.031373,1,0.003922,255,27,0.105882,224,0.878431,3,0.011765,1,0.003922,255
4,NOVOTEL,117,0.087118,0,0.000000,0.000000,37,0.316239,0.027550,57,0.487179,0.042442,15,0.128205,0.011167,2,0.017094,0.001489,4,0.034188,0.002978,2,0.017094,0.001489,5,0.042735,102,0.871795,7,0.059829,3,0.025641,117,1,0.008547,107,0.914530,9,0.076923,0,0.000000,117


In [110]:
hotel_brand_feautres.to_excel("../Output/hotel_brand_data.xlsx", index = False)